### This file uses the free version of `Unstructured` library. For more details, visit <a href="https://www.unstructured.io">unstructured.io</a>

System wide libraries required for unstructured - 
- Poppler (poppler-utils): Handles PDF processing. Extracts text, images and metadata from PDF.
- Tesseract (tesseract-ocr): ORC engine. Helps with scanned docs, images with text and PDFs which are essentially pictures.
- libmagic: File type detection library. Does so by analyzing file content, not just the extension.

In [1]:
# %pip install -Uq "unstructured[all-docs]"
# %pip install -Uq langchain_chroma
# %pip install -Uq langchain langchain-community langchain-openai
# %pip install -Uq python_dotenv

In [2]:
import json
from typing import List

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
def partition_document(file_path:str):
    """Extract elements from PDF using unstructured."""
    print("Partitioning Document...")
    
    elements = partition_pdf(
        filename=file_path, # path of pdf file
        strategy="hi_res", # high resolution - slower but more accurate
        infer_table_structure=True, # keeps tables as structured HTML
        extract_image_block_types=["Image"], # grabs images found in pdf
        extract_image_block_to_payload=True # stores images as base64 data you can actually use
    )
    
    print(f"Extracted {len(elements)} elements.")
    return elements

# file_path = "./docs/attention_is_all_you_need.pdf"
# elements = partition_document(file_path)

In [4]:
# Type of elements extracted by unstructured library
# set([str(type(el)) for el in elements])

In [5]:
# # Gathering all images
# images = [element for element in elements if element.category == 'Image']
# print(f"Found {len(images)} images")

# images[0].to_dict()
# # Open https://codebeautify.org/base64-to-image-converter and paste image_base64 value from below to see the image

In [6]:
# tables = [element for element in elements if element.category == 'Table']
# print(f"Found {len(tables)} tables")

# tables[0].to_dict()
# # Open https://jsfiddle.net/ and paste text_as_html value from below to see the table

In [7]:
def create_chunk_by_title(elements):
    """Create intelligent chunks by using title-based strategy"""
    print("Creating smaller chunks...")
    
    
    chunks = chunk_by_title(
        elements, # parsed pdf elements from previous step
        max_characters=3000, # hard limit of 3000 characters per chunk
        new_after_n_chars=2400, # try starting a new chunk after 2400 characters
        combine_text_under_n_chars=500 # merge tiny chunks below 500 characters with new chunks
    )
    
    print(f"Created {len(chunks)} chunks.")
    return chunks

# Create chunks 
# chunks = create_chunk_by_title(elements)

In [8]:
# set([str(type(chunk)) for chunk in chunks])

In [9]:
# display(chunks[4].to_dict())
# display(chunks[4].metadata.orig_elements) # the original unstructured Document class elements

In [10]:
def separate_content_types(chunk):
    """Analyze what type of contents are there in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # check for tables & images
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
                
            # handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
                    
    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text:str, tables:List[str], images: List[str]) -> str:
    """Create AI enhanced summary for mixed content"""
    try:
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        
        prompt_text = f"""You are creating a searchable description for document content retrieval.
        
        CONTENT TO ANALYZE:
        TEXT CONTENT: 
        {text}
        
        """
        
        if tables:
            prompt_text += "\nTABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
                
                prompt_text += """
                YOUR TASK: 
                Generate a comprehensive, searchable description that covers:
                
                1. Key facts, numbers and data points from the text and tables.
                2. Main topics and concepts discussed
                3. Questions this context could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms user might use
                
                Make it detailed and searchable - prioritize find-ability over brevity.
                
                SEARCHABLE DESCRIPTION:
                """
                
        # Build message content starting with texts
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add image to messages
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
            
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
    
    except Exception as err:
        print(f"    AI Summary failed: {err}")
        
        summary = f"{text[:300]}..."
        if tables:
            summary += f"[Contains {len(tables)} table(s)]"
        if images: 
            summary += f"[Contains {len(images)} image(s)]"
        
        return summary

def summarize_chunks(chunks):
    """Process all chunks with AI summaries"""
    print("Processing all chunks with AI summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i+1
        print(f"    Processing chunk {current_chunk}/{total_chunks}")
        
        content_data = separate_content_types(chunk)
        
        print(f"        Types found: {content_data['types']}")
        print(f"        Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        if content_data['tables'] or content_data['images']:
            print("     -> Creating AI Summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images']
                )
                
                print(f"    -> AI summary created successfully")
                print(f"    -> Enhanced content preview:\n\t\t {enhanced_content[:200]}.")
            except Exception as err:
                print(f"    -> AI summary failed: {err}")
                enhanced_content = content_data['text']
        else:
            print(f"    -> Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create langchain document with rich metadata
        doc = Document(
            page_content=enhanced_content, 
            metadata = {
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
        
    print(f"Processed {len(langchain_documents)} chunks.")
    return langchain_documents

# Process data into chunks using AI
# processed_chunks = summarize_chunks(chunks)

In [11]:
def export_chunks_to_json(chunks, filename="chunk_export.json"):
    """Export processed chunks to json"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i+1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        
        export_data.append(chunk_data)
        
    # save to file
    with open(filename, 'w', encoding="utf-8") as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
        
    print(f"Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
# json_data = export_chunks_to_json(chunks=processed_chunks)

In [12]:
# Creates and Persists a Chroma DB vector storage
def create_vector_store(documents, persist_directory="db/mmr_store"):
    """Creates and persists a ChromaDB vector store"""
    print("Creating embeddings and storing in ChromaDB...")
    
    embeddings_model = OpenAIEmbeddings(
        model="text-embedding-3-small"
    )
    
    print("----Creating a vector storage----")
    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=embeddings_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    
    print("----Finished creating vector store----")
    print(f"Vector store created and persisted at {persist_directory}")
    
    return vector_store

# db = create_vector_store(processed_chunks)

In [13]:
# # Retrieval

# query = "What are the two main components of the Transformer architecture?"
# retriever = db.as_retriever(search_kwargs={"k": 3})
# chunks = retriever.invoke(query)

# export_chunks_to_json(chunks, "rag_result.json")

In [14]:
def generate_final_answer(chunks, query):
    """Generate final answer using multimodal context"""
    
    try:
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        
        prompt_text = f"""Based on the following documents, please answer this question: {query}
        
        CONTEXT TO ANALYZE:
        """
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document: {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                    
                # Add tables as HTML
                tables_html = original_data.get("tables_html", "")
                if tables_html:
                    prompt_text += f"TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table: {j+1}:\n{table}\n\n"
                        
        prompt_text += '\n'
        prompt_text += """
        Please provide a clear, comprehensive answer using text, tables and images above. If document does not contain sufficient information to answer the question then say "I don't have enough information to answer that question"
        ANSWER:"""
        
        # Build a message starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from the chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("image_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
                        
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
            
        return response.content
        
    except Exception as err:
        print(f"Answer generation failed: {err}")
        return "Sorry, I encountered an error while generating the answer."
    
# final_answer = generate_final_answer(chunks, query)
# print(final_answer)

In [17]:
def run_complete_ingestion_pipeline(pdf_path:str):
    """Run the complete RAG ingestion pipeline"""
    print("Starting RAG ingestion pipeline...")
    print("="*50)
    
    # Step 1: Partition
    elements = partition_document(pdf_path)
    # Step 2: Chunking
    chunks = create_chunk_by_title(elements)
    # Step 3: AI Summarization
    summarized_chunks = summarize_chunks(chunks)
    # Step 4: Vector Storage
    db = create_vector_store(summarized_chunks, persist_directory="db/chroma_db")
    
    print("Pipeline completed successfully!")
    return db

In [19]:
db = run_complete_ingestion_pipeline("docs/attention_is_all_you_need.pdf")

Starting RAG ingestion pipeline...
Partitioning Document...


No languages specified, defaulting to English.


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

Extracted 266 elements.
Creating smaller chunks...
Created 33 chunks.
Processing all chunks with AI summaries...
    Processing chunk 1/33
        Types found: ['text']
        Tables: 0, Images: 0
    -> Using raw text (no tables/images)
    Processing chunk 2/33
        Types found: ['text']
        Tables: 0, Images: 0
    -> Using raw text (no tables/images)
    Processing chunk 3/33
        Types found: ['text']
        Tables: 0, Images: 0
    -> Using raw text (no tables/images)
    Processing chunk 4/33
        Types found: ['text']
        Tables: 0, Images: 0
    -> Using raw text (no tables/images)
    Processing chunk 5/33
        Types found: ['text', 'image']
        Tables: 0, Images: 1
     -> Creating AI Summary for mixed content...
    -> AI summary created successfully
    -> Enhanced content preview:
		 **Searchable Description:**

This document section discusses the architecture of neural sequence transduction models, focusing on the encoder-decoder structure. It h

In [23]:
query = "How many layers does the base Transformer model uses in both encoder and decoder?"
retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""
    
    try:
        # Select the LLM model
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        # Build the text prompt
        prompt_text = f"""
        Based on the following documents, please answer this question: {query}
        
        CONTENT TO ANALYZE:
        """
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"---- Document {i+1} ----\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # add tables as HTML
                tables_html = original_data.get("tables_html", "")
                if tables_html:
                    prompt_text += f"TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
                        
            prompt_text += "\n"
        prompt_text += """
        Please provide a clear, comprehensive answer using the text, tables, and images above. If the document does not contain sufficient information to answer the question, just say "I don't have enough information to answer this question"
        ANSWER:"""
        
        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("image_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
                    
        # Send to AI to get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
                
    except Exception as e:
        print(f"Answer generation failed!\n Exception raised: {e}")
        return "Sorry I encountered an error while generating the answer!"
    
# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)

The base Transformer model uses 6 layers in both the encoder and the decoder.
